In [1]:
!pip install datasets transformers torch

In [2]:
from datasets import load_dataset

issues_dataset = load_dataset("lewtun/github-issues", split="train")
issues_dataset

/home/federicovareika/Facultad/Intercambio/CBD/orange-jumpsuit/notebook-example/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Repo card metadata block was not found. Setting CardData to empty.


Dataset({
    features: ['url', 'repository_url', 'labels_url', 'comments_url', 'events_url', 'html_url', 'id', 'node_id', 'number', 'title', 'user', 'labels', 'state', 'locked', 'assignee', 'assignees', 'milestone', 'comments', 'created_at', 'updated_at', 'closed_at', 'author_association', 'active_lock_reason', 'pull_request', 'body', 'timeline_url', 'performed_via_github_app', 'is_pull_request'],
    num_rows: 3019
})

In [3]:
issues_dataset = issues_dataset.filter(
    lambda x: (x["is_pull_request"] == False and len(x["comments"]) > 0)
)
issues_dataset


Dataset({
    features: ['url', 'repository_url', 'labels_url', 'comments_url', 'events_url', 'html_url', 'id', 'node_id', 'number', 'title', 'user', 'labels', 'state', 'locked', 'assignee', 'assignees', 'milestone', 'comments', 'created_at', 'updated_at', 'closed_at', 'author_association', 'active_lock_reason', 'pull_request', 'body', 'timeline_url', 'performed_via_github_app', 'is_pull_request'],
    num_rows: 808
})

In [4]:
columns = issues_dataset.column_names
columns_to_keep = ["title", "body", "html_url", "comments"]
columns_to_remove = set(columns_to_keep).symmetric_difference(columns)
issues_dataset = issues_dataset.remove_columns(columns_to_remove)
issues_dataset

Dataset({
    features: ['html_url', 'title', 'comments', 'body'],
    num_rows: 808
})

In [5]:
issues_dataset.set_format("pandas")
df = issues_dataset[:]    

In [6]:
df["comments"][0].tolist()

['Cool, I think we can do both :)',
 '@lhoestq now the 2 are implemented.\r\n\r\nPlease note that for the the second protection, finally I have chosen to protect the master branch only from **merge commits** (see update comment above), so no need to disable/re-enable the protection on each release (direct commits, different from merge commits, can be pushed to the remote master branch; and eventually reverted without messing up the repo history).']

In [7]:
comments_df = df.explode("comments", ignore_index=True)
comments_df.head(4)

,html_url,title,comments,body
0,https://github.com/huggingface/datasets/issues...,Protect master branch,"Cool, I think we can do both :)",After accidental merge commit (91c55355b634d0d...
1,https://github.com/huggingface/datasets/issues...,Protect master branch,@lhoestq now the 2 are implemented.\r\n\r\nPle...,After accidental merge commit (91c55355b634d0d...
2,https://github.com/huggingface/datasets/issues...,Backwards compatibility broken for cached data...,Hi ! I guess the caching mechanism should have...,## Describe the bug\r\nAfter upgrading to data...
3,https://github.com/huggingface/datasets/issues...,Backwards compatibility broken for cached data...,"If it's easy enough to implement, then yes ple...",## Describe the bug\r\nAfter upgrading to data...


In [8]:
from datasets import Dataset

comments_dataset = Dataset.from_pandas(comments_df)
comments_dataset

Dataset({
    features: ['html_url', 'title', 'comments', 'body'],
    num_rows: 2964
})

In [9]:
comments_dataset = comments_dataset.map(
    lambda x: {"comment_length": len(x["comments"].split())}
)

Map: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2964/2964 [00:00<00:00, 8109.50 examples/s]


In [10]:
def concatenate_text(examples):
    return {
        "text": examples["title"]
        + " \n "
        + examples["body"]
        + " \n "
        + examples["comments"]
    }


comments_dataset = comments_dataset.map(concatenate_text)

Map: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2964/2964 [00:00<00:00, 5717.95 examples/s]


In [11]:
from transformers import AutoTokenizer, AutoModel

model_ckpt = "sentence-transformers/multi-qa-mpnet-base-dot-v1"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
model = AutoModel.from_pretrained(model_ckpt)

Loading weights: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 3787.79it/s]
MPNetModel LOAD REPORT from: sentence-transformers/multi-qa-mpnet-base-dot-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [17]:
import torch

device = torch.device("cpu")
model.to(device)

MPNetModel(
  (embeddings): MPNetEmbeddings(
    (word_embeddings): Embedding(30527, 768, padding_idx=1)
    (position_embeddings): Embedding(514, 768, padding_idx=1)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): MPNetEncoder(
    (layer): ModuleList(
      (0-11): 12 x MPNetLayer(
        (attention): MPNetAttention(
          (attn): MPNetSelfAttention(
            (q): Linear(in_features=768, out_features=768, bias=True)
            (k): Linear(in_features=768, out_features=768, bias=True)
            (v): Linear(in_features=768, out_features=768, bias=True)
            (o): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (intermediate): MPNetIntermediate(
          (dense): Linear(in_

In [18]:
def cls_pooling(model_output):
    return model_output.last_hidden_state[:, 0]

In [19]:
def get_embeddings(text_list):
    encoded_input = tokenizer(
        text_list, padding=True, truncation=True, return_tensors="pt"
    )
    encoded_input = {k: v.to(device) for k, v in encoded_input.items()}
    model_output = model(**encoded_input)
    return cls_pooling(model_output)

In [20]:
embedding = get_embeddings(comments_dataset["text"][0])
embedding.shape

torch.Size([1, 768])

In [21]:
embeddings_dataset = comments_dataset.map(
    lambda x: {"embeddings": get_embeddings(x["text"]).detach().cpu().numpy()[0]}
)

Map: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2964/2964 [41:16<00:00,  1.20 examples/s]


In [71]:
import numpy as np

# Extract the embeddings into a 2D numpy array
print("Converting dataset embeddings to NumPy array...")
corpus_embeddings = np.array(embeddings_dataset["embeddings"]).astype(np.float32)

print(f"Corpus shape: {corpus_embeddings.shape}") # Should be (2964, 768)

Converting dataset embeddings to NumPy array...
Corpus shape: (2964, 768)


In [72]:
question = "How can I load a dataset offline?"
question_embedding = get_embeddings([question]).cpu().detach().numpy().astype(np.float32)

# Index Parameters
d = corpus_embeddings.shape[1]  # 768 for MPNet
m = 8                           # 768 / 8 = 96 (Subvector size)
nbits = 8                       # 256 centroids
k = 5                           # Top 5 results


In [73]:
import faiss
import time
from orange_jumpsuit import IndexPQ as JumpsuitPQ

# ==========================================
# 1. FAISS BENCHMARK
# ==========================================
print("--- Building FAISS Index ---")
faiss_index = faiss.IndexPQ(d, m, nbits)

t0 = time.time()
faiss_index.train(corpus_embeddings)
faiss_index.add(corpus_embeddings)
faiss_build_time = time.time() - t0

t0 = time.time()
faiss_distances, faiss_indices = faiss_index.search(question_embedding, k)
faiss_search_time = time.time() - t0

--- Building FAISS Index ---


WARNING clustering 2964 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 2964 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 2964 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 2964 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 2964 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 2964 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 2964 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 2964 points to 256 centroids: please provide at least 9984 training points


In [74]:
# ==========================================
# 2. JUMPSUIT BENCHMARK
# ==========================================
print("--- Building Jumpsuit Index ---")
jumpsuit_index = JumpsuitPQ(dimension=d, n_subvectors=m, n_bits_per_value=nbits)

t0 = time.time()
jumpsuit_index.train(corpus_embeddings)
jumpsuit_index.add(corpus_embeddings)
jump_build_time = time.time() - t0

t0 = time.time()
jump_distances, jump_indices = jumpsuit_index.search(question_embedding, k)
jump_search_time = time.time() - t0

--- Building Jumpsuit Index ---


In [75]:
import pandas as pd

results = {
    "Metric": ["Build & Train Time (s)", "Search Time (s)"],
    "FAISS": [faiss_build_time, faiss_search_time],
    "Jumpsuit": [jump_build_time, jump_search_time]
}
df = pd.DataFrame(results).set_index("Metric")
print("=== PERFORMANCE COMPARISON ===")
display(df.round(4))

=== PERFORMANCE COMPARISON ===


,FAISS,Jumpsuit
Metric,,
Build & Train Time (s),0.8922,0.7540
Search Time (s),0.0004,0.0033


In [76]:
def print_results(engine_name, indices, distances):
    print(f"\n{'='*60}")
    print(f"=== TOP 5 RESULTS: {engine_name.upper()} ===")
    print(f"{'='*60}\n")
    
    for rank, (idx, dist) in enumerate(zip(indices[0], distances[0])):
        row = embeddings_dataset[int(idx)]
        print(f"Rank: {rank + 1} | Score (Distance): {dist:.4f}")
        print(f"TITLE: {row['title']}")
        print(f"URL: {row['html_url']}")
        print(f"COMMENT: {row['comments']}")
        print("-" * 50)

# Print FAISS Results
print_results("FAISS", faiss_indices, faiss_distances)

# Print Jumpsuit Results
print_results("Jumpsuit", jump_indices, jump_distances)


=== TOP 5 RESULTS: FAISS ===

Rank: 1 | Score (Distance): 23.4873
TITLE: Discussion using datasets in offline mode
URL: https://github.com/huggingface/datasets/issues/824
COMMENT: No comments ?
--------------------------------------------------
Rank: 2 | Score (Distance): 23.4873
TITLE: Discussion using datasets in offline mode
URL: https://github.com/huggingface/datasets/issues/824
COMMENT: I think it would be very cool. I'm currently working on a cluster from Compute Canada, and I have internet access only when I'm not in the nodes where I run the scripts. So I was expecting to be able to use the wmt14 dataset until I realized I needed internet connection even if I downloaded the data already. I'm going to try option 2 you mention for now though! Thanks ;)
--------------------------------------------------
Rank: 3 | Score (Distance): 23.4873
TITLE: Discussion using datasets in offline mode
URL: https://github.com/huggingface/datasets/issues/824
COMMENT: Requiring online connection i